# Valoricore × LlamaIndex

**Deterministic vector memory with cryptographic audit trails — natively in LlamaIndex.**

| Section | What you'll see |
|---|---|
| 1–4 | Install, imports, embedder, one-line init |
| 5–6 | VectorStoreIndex + retriever |
| 7 | **Index comparison — BruteForce vs HNSW** |
| 8 | Low-level node add / query |
| 9 | Delete a record |
| 10 | Score semantics (0, 1] |
| 11–12 | State hash + Snapshot/restore |

[![GitHub](https://img.shields.io/badge/GitHub-Valori--Kernel-6c47ff?logo=github)](https://github.com/varshith-Git/Valori-Kernel)

## 1 · Install

Three packages required:
- `valoricore[llamaindex]` — core SDK + llama-index-core
- `llama-index-embeddings-huggingface` — HuggingFaceEmbedding class
- `sentence-transformers` — model backend (offline, no API key)

In [ ]:
%%capture
!pip install "valoricore[llamaindex]" llama-index-embeddings-huggingface sentence-transformers

## 2 · Imports

In [ ]:
import time, statistics
from valoricore.integrations import ValoricoreLlamaIndex

from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.core.schema import TextNode, Document
from llama_index.core.vector_stores.types import VectorStoreQuery

print("Imports OK ✓")

## 3 · Embedder

Offline `sentence-transformers` model — no API key. Swap with `OpenAIEmbedding()` for production.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.embed_model = embed_model
# Settings.llm is not set — this notebook uses retriever-only mode.
# For full RAG with an LLM see the commented block in section 6.

dim = len(embed_model.get_text_embedding("test"))
print(f"Embedding dim: {dim}")

## 4 · One-line initialization

In [ ]:
vector_store = ValoricoreLlamaIndex(
    path       = "/tmp/valori_llamaindex_demo",
    index_kind = "bruteforce",   # changed per-section below
)
print(f"Store initialized ✓  |  stores_text={vector_store.stores_text}")

## 5 · Build a VectorStoreIndex from documents

In [ ]:
documents = [
    Document(text="Valoricore is a deterministic vector database built in Rust."),
    Document(text="Q16.16 fixed-point arithmetic produces identical results on x86, ARM, and RISC-V."),
    Document(text="Every insert is backed by a BLAKE3 Merkle proof."),
    Document(text="The event log is append-only and fsynced before any live state change."),
    Document(text="Crash recovery is mathematically proven, not just claimed."),
    Document(text="Leader-follower replication uses tokio::sync::watch for race-free coordination."),
    Document(text="LlamaIndex users can swap Chroma for Valoricore with one import."),
    Document(text="The HNSW index enables sub-millisecond search at millions of records."),
]

ctx   = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(documents, storage_context=ctx, show_progress=True)
print(f"\nIndex built | hash: {vector_store.get_state_hash()[:16]}...")

## 6 · Retriever mode (no LLM required)

In [ ]:
retriever = index.as_retriever(similarity_top_k=3)
nodes     = retriever.retrieve("How does crash recovery work?")

print(f"Retrieved {len(nodes)} nodes:\n")
for i, n in enumerate(nodes, 1):
    print(f"{i}. score={n.score:.4f}  →  {n.text}")

In [ ]:
# ── Full RAG with OpenAI LLM ─────────────────────────────────────────────────
# Requires: pip install llama-index-llms-openai
#
# from llama_index.llms.openai  import OpenAI
# from llama_index.embeddings.openai import OpenAIEmbedding
# import os; os.environ["OPENAI_API_KEY"] = "sk-..."
#
# Settings.llm         = OpenAI(model="gpt-4o-mini")
# Settings.embed_model = OpenAIEmbedding()
#
# openai_store = ValoricoreLlamaIndex(path="/tmp/valori_openai")
# ctx2  = StorageContext.from_defaults(vector_store=openai_store)
# idx2  = VectorStoreIndex.from_documents(documents, storage_context=ctx2)
# resp  = idx2.as_query_engine(similarity_top_k=4).query(
#     "How does Valoricore prove crash recovery?"
# )
# print(resp)

---
## 7 · Index Comparison — BruteForce vs HNSW

### Architecture of each index inside Valoricore

**BruteForce** (`src/index/brute_force.rs`)
- Stateless: no graph, no clusters, no build phase
- On every query: scans every active record in the pool
- Distance: Q16.16 fixed-point L2², deterministic on all hardware
- Recall: **100% exact** — mathematically guaranteed
- Complexity: O(n) per query

**HNSW** (`src/hnsw.rs`)
- Multi-layer proximity graph with flat arena storage
- Parameters (from Rust source): `M=16`, `M_MAX=32`, `EF_CONSTRUCTION=64`
- Distance: Q16.16 fixed-point — results are **bit-identical** across x86, ARM, RISC-V
- Recall: ~95%+ approximate nearest neighbor
- Complexity: O(log n) per query after O(n log n) build

> **IVF status:** IVF is on the roadmap. The kernel has the deterministic k-means
> infrastructure (FNV-1a seeding, i64 arithmetic). The index wrapper is not yet
> exposed via the Python API. Use `bruteforce` or `hnsw` for current deployments.

In [ ]:
# Build both index types with identical data
corpus_texts = [
    "Deterministic fixed-point arithmetic for reproducible AI.",
    "BLAKE3 cryptographic proofs on every vector insert.",
    "HNSW proximity graph with M=16 bidirectional links.",
    "BruteForce exact scan — 100% recall guaranteed.",
    "Append-only event log with fsync before live apply.",
    "Tenant isolation via write-time integer tags.",
    "Snapshot and restore with bit-exact hash verification.",
    "Knowledge graph traversal over typed edges.",
    "Q16.16 format: 16 integer bits, 16 fractional bits in i32.",
    "Leader-follower replication coordinated by watch channels.",
]

def _make_nodes(texts):
    return [
        TextNode(
            text      = t,
            id_       = f"node-{i}",
            embedding = embed_model.get_text_embedding(t),
        )
        for i, t in enumerate(texts)
    ]

nodes = _make_nodes(corpus_texts)   # embed once, reuse for both stores

bf_store   = ValoricoreLlamaIndex(path="/tmp/valori_li_bf",   index_kind="bruteforce")
hnsw_store = ValoricoreLlamaIndex(path="/tmp/valori_li_hnsw", index_kind="hnsw")

bf_store.add(nodes)
hnsw_store.add(nodes)

print(f"BruteForce — {len(nodes)} nodes inserted")
print(f"HNSW       — {len(nodes)} nodes inserted")

In [ ]:
# Benchmark across test queries
test_queries = [
    "cryptographic hash and audit trail",
    "approximate nearest neighbor graph traversal",
    "crash recovery and event sourcing",
    "tenant filtering and namespace isolation",
    "deterministic reproducible search results",
    "snapshot restore and bit-exact verification",
]
K = 3

bf_times, hnsw_times, recalls = [], [], []

for q in test_queries:
    qvec  = embed_model.get_text_embedding(q)
    query = VectorStoreQuery(query_embedding=qvec, similarity_top_k=K)

    t0     = time.perf_counter()
    gt     = bf_store.query(query)
    bf_times.append((time.perf_counter() - t0) * 1000)

    t0     = time.perf_counter()
    ap     = hnsw_store.query(query)
    hnsw_times.append((time.perf_counter() - t0) * 1000)

    gt_ids = set(gt.ids)
    ap_ids = set(ap.ids)
    recalls.append(len(gt_ids & ap_ids) / K)

print("── LlamaIndex Index Benchmark ────────────────────────────────")
print(f"  {'Index':<12} {'Avg (ms)':>10} {'Max (ms)':>10}")
print(f"  {'bruteforce':<12} {statistics.mean(bf_times):>10.3f} {max(bf_times):>10.3f}")
print(f"  {'hnsw':<12} {statistics.mean(hnsw_times):>10.3f} {max(hnsw_times):>10.3f}")
print(f"\n  HNSW recall vs BruteForce ground truth: {statistics.mean(recalls)*100:.1f}%")

print("\n── Per-query recall ──────────────────────────────────────────")
for q, r in zip(test_queries, recalls):
    bar = '█' * int(r * 10) + '░' * (10 - int(r * 10))
    print(f"  {bar}  {r*100:5.1f}%  '{q}'")

In [ ]:
# Score comparison — BruteForce vs HNSW on the same query
demo_q = VectorStoreQuery(
    query_embedding  = embed_model.get_text_embedding("BLAKE3 cryptographic proof"),
    similarity_top_k = 3,
)

gt_r = bf_store.query(demo_q)
ap_r = hnsw_store.query(demo_q)

print("Query: 'BLAKE3 cryptographic proof'\n")
print(f"{'BruteForce (exact)':^60}   {'HNSW (approximate)':^60}")
print("-" * 60 + "   " + "-" * 60)

for (gn, gs, gi), (an, as_, ai) in zip(
    zip(gt_r.nodes, gt_r.similarities, gt_r.ids),
    zip(ap_r.nodes, ap_r.similarities, ap_r.ids),
):
    match  = "✓" if gi == ai else "≠"
    bf_col = f"sim={gs:.4f}  {gn.text[:45]}"
    hn_col = f"sim={as_:.4f}  {an.text[:45]}"
    print(f"{bf_col:<60}   {hn_col:<60}  {match}")

In [ ]:
# Verify HNSW scores are in (0, 1] — the LlamaIndex contract
for sim in ap_r.similarities:
    assert 0.0 < sim <= 1.0, f"Score out of range: {sim}"
    print(f"  sim={sim:.6f}  ∈ (0, 1] ✓")

### Index decision guide

```
How many vectors?
  < 50K   → bruteforce  (exact, no build cost, provable recall)
  > 100K  → hnsw        (sub-ms latency, ~95%+ recall)
  50–100K → bruteforce until latency is a concern, then migrate

What matters most?
  Must prove recall to a regulator  → bruteforce (100% by definition)
  Throughput in production RAG      → hnsw
  Offline batch processing          → either (latency rarely matters)
```

---
## 8 · Low-level node add / query

In [ ]:
raw_store = ValoricoreLlamaIndex(path="/tmp/valori_li_raw")

raw_texts = [
    "IVF uses deterministic k-means with FNV-1a seeding.",
    "BLAKE3 is a cryptographic hash with parallel tree hashing.",
    "Q16.16: 16 integer bits + 16 fractional bits in i32.",
]
raw_nodes = [
    TextNode(text=t, id_=f"raw-{i}",
             embedding=embed_model.get_text_embedding(t))
    for i, t in enumerate(raw_texts)
]

ids = raw_store.add(raw_nodes)
print(f"Inserted: {ids}")

result = raw_store.query(VectorStoreQuery(
    query_embedding  = embed_model.get_text_embedding("BLAKE3 hash"),
    similarity_top_k = 2,
))
for node, sim, nid in zip(result.nodes, result.similarities, result.ids):
    print(f"  {nid}  sim={sim:.4f}  →  {node.text}")

## 9 · Delete a record

`ValoricoreLlamaIndex` does not maintain a `node_id → record_id` secondary index,
so LlamaIndex's `delete(ref_doc_id)` is a no-op at the vector store level.
Use `store._client.delete(record_id)` for hard deletion — track the integer
`record_id` at insert time via `record_count()`.

In [ ]:
del_store = ValoricoreLlamaIndex(path="/tmp/valori_li_delete")

node = TextNode(
    text="This record will be deleted.",
    id_="del-node",
    embedding=embed_model.get_text_embedding("This record will be deleted."),
)
del_store.add([node])

record_id   = del_store._client.record_count() - 1   # last inserted
hash_before = del_store.get_state_hash()

del_store._client.delete(record_id)   # real hard delete via MemoryClient

hash_after = del_store.get_state_hash()
print(f"record_id    : {record_id}")
print(f"Hash before  : {hash_before[:24]}...")
print(f"Hash after   : {hash_after[:24]}...")
print(f"Hash changed : {hash_before != hash_after} ✓")

## 10 · Score semantics — `(0, 1]`

Valoricore converts raw Q16.16² L2 distance → similarity via `1/(1+dist)`.
Score = 1.0 means identical vectors. LlamaIndex expects this range.

In [ ]:
for node, sim in zip(result.nodes, result.similarities):
    assert 0.0 < sim <= 1.0
    print(f"  sim={sim:.6f}  ∈ (0, 1] ✓  →  {node.text[:55]}")

## 11 · Cryptographic state hash

In [ ]:
h1 = vector_store.get_state_hash()
vector_store.add([TextNode(
    text="post-hash insert", id_="ph",
    embedding=embed_model.get_text_embedding("post-hash insert")
)])
h2 = vector_store.get_state_hash()
print(f"Before: {h1}")
print(f"After : {h2}")
print(f"Differ: {h1 != h2} ✓")

## 12 · Snapshot & bit-exact restore

In [ ]:
h_before = vector_store.get_state_hash()
snap     = vector_store.snapshot()

restored = ValoricoreLlamaIndex(path="/tmp/valori_li_restored")
restored.restore(snap)

h_after = restored.get_state_hash()
print(f"Snapshot : {len(snap):,} bytes")
print(f"Before   : {h_before}")
print(f"After    : {h_after}")
print(f"Bit-exact: {h_before == h_after} ✓")

---
## Summary

| Feature | API |
|---|---|
| Init | `ValoricoreLlamaIndex(path=..., index_kind="bruteforce\|hnsw")` |
| Insert nodes | `store.add(nodes)` — nodes must have `node.embedding` set |
| Query | `store.query(VectorStoreQuery(query_embedding=..., similarity_top_k=k))` |
| Score range | `(0, 1]` via `1/(1+L2²)` — higher = more similar |
| Delete | `store._client.delete(record_id)` — track id at insert time |
| State proof | `store.get_state_hash()` — 64-char BLAKE3 |
| Snapshot | `store.snapshot()` / `store.restore(snap)` |

**Index quick-reference:**

| `index_kind` | Recall | Latency | Use when |
|---|---|---|---|
| `bruteforce` | 100% exact | O(n) | < 50K vectors, compliance, CI |
| `hnsw` | ~95%+ ANN | O(log n) | > 100K vectors, production |

**Other notebooks:**
- [End-to-End Demo](https://colab.research.google.com/drive/1QO1yQMQoGbp9fwrb00KVKTq5bYVGXgJv)
- [LangChain Integration](https://colab.research.google.com/drive/1HezK4l-Hbc6AdHxJNLwSqAgzr8WaKhiq)
- [GitHub](https://github.com/varshith-Git/Valori-Kernel)